In [1]:
# 1. Object Detection with YOLOv5 (Using Pretrained Weights) 
# Problem: Detect objects using pre-trained YOLOv5 model. 

import torch
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

img = r"C:\Hope AI\AI Course Tamil\Question bank\6. Deep Learning Week11\6.Deep Learning (Week 11)\Batch2\1.Deep_Learning_Question_Set2\4.Coding\flower.jpg"

results = model(img)
results.print()

Using cache found in C:\Users\Raji/.cache\torch\hub\ultralytics_yolov5_master


YOLOv5  2026-9-28 Python-3.12.7 torch-2.12.1+cpu CPU

Fusing layers... 
YOLOv5s summary: 124 layers, 7,225,885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
image 1/1: 4000x4800 1 potted plant
Speed: 277.4ms pre-process, 156.7ms inference, 27.6ms NMS per image at shape (1, 3, 544, 640)


In [2]:
# 2. Preprocessing Image for ResNet Input  Problem: Write a function to preprocess image for pretrained ResNet. 

from PIL import Image 
from torchvision import transforms
import torch

def preprocess_image(image_path: str) -> torch.Tensor:
    """Load and preprocess an image for a pretrained torchvision ResNet model."""
    # Define the standard ImageNet preprocessing pipeline
    transform_pipeline = transforms.Compose([
        transforms.Resize(256),                    # Resize shorter side to 256
        transforms.CenterCrop(224),                # Crop center 224x224 pixels   
        transforms.ToTensor(),                     # Convert PIL image to [0, 1] Tensor   
        transforms.Normalize(                      # Normalize with ImageNet stats
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])
    # Load image and apply transforms
    image = Image.open(image_path).convert("RGB")
    tensor_image = transform_pipeline(image)

    # Add a batch dimension (shape becomes [1, 3, 224, 224])
    return tensor_image.unsqueeze(0)

image_path=(r"C:\Hope AI\AI Course Tamil\Question bank\6. Deep Learning Week11\6.Deep Learning (Week 11)\Batch2\1.Deep_Learning_Question_Set2\4.Coding\flower.jpg")
tensor = preprocess_image(image_path)
print(tensor.shape)

torch.Size([1, 3, 224, 224])


In [3]:
# 3. CNN for CIFAR-10 Classification.  Problem: Build a CNN model to classify CIFAR-10 images. 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()

train = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
test = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=64)

# CNN Model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 10)
        )

    def forward(self, x):
        return self.net(x)

model = CNN()

# Loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
for epoch in range(5):
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Testing
correct = total = 0

with torch.no_grad():
    for x, y in test_loader:
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Test Accuracy:", 100 * correct / total)

Epoch 1, Loss: 1.7819
Epoch 2, Loss: 0.8617
Epoch 3, Loss: 0.8901
Epoch 4, Loss: 0.7894
Epoch 5, Loss: 0.5695
Test Accuracy: 67.01


#### 4. Face Mask Detection – Real-Time Webcam Classifier.  Problem: Build a real-time face mask detector using a trained CNN model and OpenCV.

In [5]:
# 5. Custom Transfer Learning with Frozen + Trainable Layers  Problem: Load a pretrained MobileNetV2 model, freeze base layers, 
# add custom classifier, and fine-tune the top few layers.

import tensorflow as tf
from tensorflow.keras import layers, models

# Load dataset
train = tf.keras.utils.image_dataset_from_directory("dataset/train", image_size=(224, 224), batch_size=32)

val = tf.keras.utils.image_dataset_from_directory("dataset/val", image_size=(224, 224), batch_size=32)

test = tf.keras.utils.image_dataset_from_directory("dataset/test", image_size=(224, 224), batch_size=32)

# Pretrained MobileNetV2
base = tf.keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base
base.trainable = False

# Custom classifier
model = models.Sequential([base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(2, activation="softmax")
])

# Train classifier
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(train, validation_data=val, epochs=3)

# Fine-tune top 20 layers
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(train, validation_data=val, epochs=2)

# Final evaluation
model.evaluate(test)

Found 19943 files belonging to 2 classes.
Found 2492 files belonging to 2 classes.
Found 2495 files belonging to 2 classes.


Epoch 1/3
624/624 ━━━━━━━━━━━━━━━━━━━━ 509s 809ms/step - accuracy: 0.7151 - loss: 0.5597 - val_accuracy: 0.7452 - val_loss: 0.5083
Epoch 2/3
624/624 ━━━━━━━━━━━━━━━━━━━━ 469s 752ms/step - accuracy: 0.7527 - loss: 0.5005 - val_accuracy: 0.7508 - val_loss: 0.5023
Epoch 3/3
624/624 ━━━━━━━━━━━━━━━━━━━━ 467s 749ms/step - accuracy: 0.7740 - loss: 0.4732 - val_accuracy: 0.7612 - val_loss: 0.4909
Epoch 1/2
624/624 ━━━━━━━━━━━━━━━━━━━━ 575s 910ms/step - accuracy: 0.7851 - loss: 0.4502 - val_accuracy: 0.5823 - val_loss: 1.6834
Epoch 2/2
624/624 ━━━━━━━━━━━━━━━━━━━━ 560s 898ms/step - accuracy: 0.8604 - loss: 0.3158 - val_accuracy: 0.7311 - val_loss: 0.6990
78/78 ━━━━━━━━━━━━━━━━━━━━ 56s 716ms/step - accuracy: 0.7311 - loss: 0.7025


[0.7024779319763184, 0.731062114238739]